# NB_hospital_daily_summary

Builds `adb_caresync_smd.gold.hospital_daily_summary` — one row per hospital per day with test volumes, patient counts, abnormal and critical rates, and day-over-day trend metrics.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync_smd.gold.hospital_daily_summary AS

WITH daily_base AS (
  SELECT
    lr.hospital_id,
    lr.test_date                                                AS summary_date,
    COUNT(*)                                                    AS total_tests,
    COUNT(DISTINCT lr.patient_id)                               AS unique_patients_tested,
    COUNT(DISTINCT lr.doctor_id)                                AS active_doctor_count,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)   AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)   AS critical_test_count
  FROM adb_caresync_smd.silver.lab_results lr
  GROUP BY lr.hospital_id, lr.test_date
), enriched AS (
  SELECT
    db.hospital_id,
    h.hospital_name,
    db.summary_date,
    c.day_name,
    c.is_weekend,
    db.total_tests,
    db.unique_patients_tested,
    db.active_doctor_count,
    db.abnormal_test_count,
    db.critical_test_count,
    ROUND(db.abnormal_test_count / NULLIF(db.total_tests, 0) * 100, 2)
                                                                AS abnormal_rate_pct,
    ROUND(db.critical_test_count / NULLIF(db.total_tests, 0) * 100, 2)
                                                                AS critical_rate_pct
  FROM daily_base db
  LEFT JOIN adb_caresync_smd.silver.hospitals h
    ON db.hospital_id = h.hospital_id
  LEFT JOIN adb_caresync_smd.silver.calendar c
    ON c.date = db.summary_date
)

SELECT
  hospital_id,
  hospital_name,
  summary_date,
  day_name,
  is_weekend,
  total_tests,
  unique_patients_tested,
  active_doctor_count,
  abnormal_test_count,
  critical_test_count,
  abnormal_rate_pct,
  critical_rate_pct,
  LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date)
                                                                AS prev_day_total_tests,
  ROUND(
    (total_tests - LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date))
    / NULLIF(LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date), 0) * 100,
    2
  )                                                             AS test_volume_change_pct
FROM enriched;

In [0]:
updated_df = spark.table("adb_caresync_smd.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))